# 4a_matrizes_transicao — matrizes de validade das transições (4 classes)

Etapa 7 do projeto redesenhado (pasta `C:\cmap_ftl\terraclass`). Gera as matrizes de transição entre cada par de datas consecutivas (2005→2006, ..., 2009→2010), no formato lido pelo CMAP (`read_transition_matrix` do repositório `CMAP_FTL`): **linhas = classe em t−1, colunas = classe em t**, primeira linha e primeira coluna com as siglas, separador `;`.

**Abordagem simplificada (válida / inválida).** Cada transição vale 1 (válida) ou 0 (inválida). Uma trajetória com alguma transição inválida recebe P(s) = 0; as demais recebem o mesmo peso. É a abordagem usada na dissertação e uma das duas propostas por Reis et al. (2020).

**Regra do projeto redesenhado.** Com a legenda de 4 classes (F, VS, PA, AG), **a única transição proibida é a volta para Floresta**: VS → F, PA → F e AG → F. Todas as outras são válidas, inclusive F → VS (desmatamento e regeneração dentro do intervalo de um ano) e as trocas entre VS, PA e AG. Como as 4 classes podem ocorrer em todas as datas, a mesma matriz vale para os 5 pares.

O notebook também confere a matriz contra o **TerraClass 2008 → 2010** da área de estudo: pixels que o TerraClass registra com volta para Floresta nunca poderão ser acertados pelo CMAP nas duas datas.

Lê `resultados\0b_conjunto_treino\<NOME_CONJUNTO>\legenda.json`, `resultados\0c_referencia_terraclass\` e o repositório `C:\cmap_ftl\CMAP_FTL` (só leitura); grava só em `resultados\4a_matrizes_transicao\`.

## Parâmetros

In [1]:
# ============================== PARÂMETROS ==============================
PASTA_PROJETO    = r"C:\cmap_ftl\terraclass"
NOME_CONJUNTO    = "6bandas"
PASTA_CONJUNTO   = rf"{PASTA_PROJETO}\resultados\0b_conjunto_treino\{NOME_CONJUNTO}"   # legenda.json
PASTA_REFERENCIA = rf"{PASTA_PROJETO}\resultados\0c_referencia_terraclass"
PASTA_REPO       = r"C:\cmap_ftl\CMAP_FTL"                    # pacote functions (só leitura)
PASTA_SAIDA      = rf"{PASTA_PROJETO}\resultados\4a_matrizes_transicao"

ANOS = [2005, 2006, 2007, 2008, 2009, 2010]

# Matriz de validade: linhas = classe em t-1, colunas = classe em t; V = válida, I = inválida.
# Só a volta para Floresta é proibida.
MATRIZ_BASE = """
       F    VS   PA   AG
F      V    V    V    V
VS     I    V    V    V
PA     I    V    V    V
AG     I    V    V    V
"""

## 1. Matriz base

In [2]:
import os, sys, json
import numpy as np
import pandas as pd
import rasterio

if not os.path.abspath(PASTA_SAIDA).lower().startswith(os.path.abspath(PASTA_PROJETO).lower() + os.sep):
    raise ValueError(f"PASTA_SAIDA precisa ficar dentro de {PASTA_PROJETO}")
os.makedirs(PASTA_SAIDA, exist_ok=True)
with open(os.path.join(PASTA_CONJUNTO, "legenda.json"), encoding="utf-8") as arq:
    legenda = json.load(arq)
CLASSES = {int(k): v for k, v in legenda["classes"].items()}     # código -> sigla
CODIGOS = sorted(CLASSES)
SIGLAS = [CLASSES[c] for c in CODIGOS]

linhas = [l.split() for l in MATRIZ_BASE.strip().splitlines()]
base = pd.DataFrame([l[1:] for l in linhas[1:]], index=[l[0] for l in linhas[1:]], columns=linhas[0])
if sorted(base.index) != sorted(SIGLAS) or sorted(base.columns) != sorted(SIGLAS):
    raise ValueError(f"A matriz base precisa ter exatamente as classes {SIGLAS} nas linhas e nas colunas")
if not base.isin(["V", "I"]).all().all():
    raise ValueError("Use apenas V ou I na matriz base")
base = (base.loc[SIGLAS, SIGLAS] == "V").astype(int)       # ordem da legenda; 1 = válida
base.index.name = "t-1 \\ t"
base.to_csv(os.path.join(PASTA_SAIDA, "matriz_base_validade.csv"), sep=";")
print("Matriz base (1 = válida):")
print(base.to_string())
print("\nConferência das regras:")
print("  alguma classe além de F vira F?", bool(base.loc[base.index != "F", "F"].any()))
print("  F pode virar qualquer classe?  ", bool(base.loc["F"].all()))
print("  transições proibidas:          ", [f"{a}->{b}" for a in SIGLAS for b in SIGLAS if base.loc[a, b] == 0])

Matriz base (1 = válida):
         F  VS  PA  AG
t-1 \ t               
F        1   1   1   1
VS       0   1   1   1
PA       0   1   1   1
AG       0   1   1   1

Conferência das regras:
  alguma classe além de F vira F? False
  F pode virar qualquer classe?   True
  transições proibidas:           ['VS->F', 'PA->F', 'AG->F']


## 2. Matrizes por par de datas

Os arquivos `transicao_AAAA_AAAA.csv` são os que entram no CMAP. O número de trajetórias válidas sai do produto das matrizes (soma de todos os caminhos).

In [3]:
matrizes, arquivos = [], []
for a1, a2 in zip(ANOS[:-1], ANOS[1:]):
    m = base.copy()
    m.index.name = None
    arq = os.path.join(PASTA_SAIDA, f"transicao_{a1}_{a2}.csv")
    m.to_csv(arq, sep=";")
    matrizes.append(m)
    arquivos.append(arq)
print(f"{len(arquivos)} matrizes gravadas ({', '.join(os.path.basename(a) for a in arquivos)})")

caminhos = np.ones((1, len(SIGLAS)), dtype=np.int64)
for m in matrizes:
    caminhos = caminhos @ m.to_numpy()
n_validas = int(caminhos.sum())
n_total = len(SIGLAS) ** len(ANOS)
print(f"Trajetórias válidas: {n_validas} de {n_total} ({100 * n_validas / n_total:.1f}%)")
print("Válidas por número de anos iniciais em Floresta (depois que sai de F, nunca volta):")
for k in range(len(ANOS), -1, -1):
    n_k = 1 if k == len(ANOS) else (len(SIGLAS) - 1) ** (len(ANOS) - k)
    print(f"  {k} ano(s) em F no início: {n_k} trajetória(s)")

5 matrizes gravadas (transicao_2005_2006.csv, transicao_2006_2007.csv, transicao_2007_2008.csv, transicao_2008_2009.csv, transicao_2009_2010.csv)
Trajetórias válidas: 1093 de 4096 (26.7%)
Válidas por número de anos iniciais em Floresta (depois que sai de F, nunca volta):
  6 ano(s) em F no início: 1 trajetória(s)
  5 ano(s) em F no início: 3 trajetória(s)
  4 ano(s) em F no início: 9 trajetória(s)
  3 ano(s) em F no início: 27 trajetória(s)
  2 ano(s) em F no início: 81 trajetória(s)
  1 ano(s) em F no início: 243 trajetória(s)
  0 ano(s) em F no início: 729 trajetória(s)


## 3. Conferência com o TerraClass 2008 → 2010

Compara as classes do TerraClass de 2008 e 2010 em cada pixel da área de estudo com as duas classes dentro da legenda. Entre 2008 e 2010 há um ano sem referência (2009): a transição é aceita se existir algum caminho válido passando por 2009. As transições que a matriz não permite aparecem na tabela; esses pixels nunca poderão ser acertados pelo CMAP nos dois anos.

In [4]:
ref = {}
for ano in (2008, 2010):
    with rasterio.open(os.path.join(PASTA_REFERENCIA, f"referencia_{ano}.tif")) as src:
        ref[ano] = src.read(1)
M = base.to_numpy()
alcance = M @ M                                   # 2008 -> 2009 -> 2010
sel = np.isin(ref[2008], CODIGOS) & np.isin(ref[2010], CODIGOS)
de = np.searchsorted(CODIGOS, ref[2008][sel])
para = np.searchsorted(CODIGOS, ref[2010][sel])
tab = pd.crosstab(pd.Categorical([SIGLAS[i] for i in de], SIGLAS), pd.Categorical([SIGLAS[j] for j in para], SIGLAS))
tab.index.name, tab.columns.name = "2008", "2010"
tab.to_csv(os.path.join(PASTA_SAIDA, "transicoes_terraclass_2008_2010.csv"), sep=";")
print("Transições no TerraClass da área de estudo (pixels):")
print(tab.to_string())
invalida = alcance[de, para] == 0
linhas = [{"de": SIGLAS[a], "para": SIGLAS[b], "pixels": int(((de == a) & (para == b) & invalida).sum())}
          for a in range(len(SIGLAS)) for b in range(len(SIGLAS)) if alcance[a, b] == 0]
tab_invalidas = pd.DataFrame(linhas)
tab_invalidas = tab_invalidas[tab_invalidas["pixels"] > 0]
tab_invalidas.to_csv(os.path.join(PASTA_SAIDA, "transicoes_invalidas_na_referencia.csv"), index=False)
print(f"\nPixels com as duas datas na legenda: {int(sel.sum())} | com transição proibida: {int(invalida.sum())} "
      f"({100 * invalida.mean():.3f}%)")
print(tab_invalidas.to_string(index=False) if len(tab_invalidas) else "Nenhuma transição proibida na referência.")

Transições no TerraClass da área de estudo (pixels):
2010       F     VS      PA     AG
2008                              
F     634437    809    1685    746
VS        11  82515    4111    356
PA        32   5358  111297   1760
AG         0    126    7996  22560

Pixels com as duas datas na legenda: 873799 | com transição proibida: 43 (0.005%)
de para  pixels
VS    F      11
PA    F      32


## 4. Conferência com o código do CMAP

Lê os arquivos gerados com as funções do repositório `CMAP_FTL` e calcula as probabilidades a priori das trajetórias. O aviso de que as linhas não somam 1 é esperado, porque são matrizes de validade (0/1). O número de trajetórias com P(s) > 0 deve ser igual ao da seção 2.

In [5]:
sys.path.insert(0, PASTA_REPO)
from functions.transitions import create_transition_matrix_table

tabela_trajetorias = create_transition_matrix_table(arquivos, output_path=os.path.join(PASTA_SAIDA, "trajetorias.csv"))
print("Confere com a seção 2:", int((tabela_trajetorias["final_weight"] > 0).sum()) == n_validas)
tabela_trajetorias[tabela_trajetorias["final_weight"] > 0].head()

Aviso: em 'transicao_2005_2006.csv', 4 linha(s) não somam 1 (Eq. 2.13) — esperado apenas em matriz de validade.
Aviso: em 'transicao_2006_2007.csv', 4 linha(s) não somam 1 (Eq. 2.13) — esperado apenas em matriz de validade.
Aviso: em 'transicao_2007_2008.csv', 4 linha(s) não somam 1 (Eq. 2.13) — esperado apenas em matriz de validade.
Aviso: em 'transicao_2008_2009.csv', 4 linha(s) não somam 1 (Eq. 2.13) — esperado apenas em matriz de validade.
Aviso: em 'transicao_2009_2010.csv', 4 linha(s) não somam 1 (Eq. 2.13) — esperado apenas em matriz de validade.
Trajetórias: 4096 | P(s) > 0: 1093 | inválidas: 3003 | P(ω1) uniforme
Confere com a seção 2: True


,Stage_1,Stage_2,Stage_3,Stage_4,Stage_5,Stage_6,weight_1,weight_2,weight_3,weight_4,weight_5,final_weight
0,F,F,F,F,F,F,1.0,1.0,1.0,1.0,1.0,1.0
1,F,F,F,F,F,VS,1.0,1.0,1.0,1.0,1.0,1.0
2,F,F,F,F,F,PA,1.0,1.0,1.0,1.0,1.0,1.0
3,F,F,F,F,F,AG,1.0,1.0,1.0,1.0,1.0,1.0
5,F,F,F,F,VS,VS,1.0,1.0,1.0,1.0,1.0,1.0


## Saídas

| Arquivo | Conteúdo |
|---|---|
| `matriz_base_validade.csv` | matriz base (1 = válida) |
| `transicao_AAAA_AAAA.csv` | matriz de cada par de datas consecutivas, lida pelo CMAP |
| `trajetorias.csv` | todas as trajetórias e P(s) (`final_weight`) |
| `transicoes_terraclass_2008_2010.csv`, `transicoes_invalidas_na_referencia.csv` | conferência com o TerraClass |